# Support Vector Machines (SVMs)

## Learning Objectives
1. Derive the hard-margin SVM objective and understand max-margin geometry using NumPy.
2. Compare SVC kernels (linear, RBF, poly) with sklearn and handle OOM for large datasets.
3. Apply SVM with TF-IDF features for text classification as a competitive baseline.
4. Visualize the kernel trick in 2D->3D and compare SVM vs Logistic Regression in high dimensions.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification, make_moons, make_circles
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Level 1: Hard-Margin SVM — Margin Geometry with NumPy

In [ ]:
# The SVM finds the hyperplane w^T x + b = 0 that maximizes the margin 2/||w||.
# Support vectors are the closest points from each class to the decision boundary.
# Hard-margin SVM: no misclassifications allowed (linearly separable data only).

def margin_width(w: np.ndarray) -> float:
    """Margin = 2 / ||w||. Maximizing margin = minimizing ||w||."""
    return 2.0 / np.linalg.norm(w)


def functional_margin(
    X: np.ndarray, y: np.ndarray, w: np.ndarray, b: float
) -> np.ndarray:
    """y_i * (w^T x_i + b): should be >= 1 for correctly classified support vectors."""
    return y * (X @ w + b)


# Generate 2D linearly separable data
rng = np.random.default_rng(0)
n_pos = n_neg = 30
X_pos = rng.normal([2, 2], 0.5, (n_pos, 2))
X_neg = rng.normal([-2, -2], 0.5, (n_neg, 2))
X_2d = np.vstack([X_pos, X_neg])
y_2d = np.array([1] * n_pos + [-1] * n_neg, dtype=float)

# Fit with sklearn to get the true maximum-margin hyperplane
from sklearn.svm import SVC as _SVC
svc_2d = _SVC(kernel='linear', C=1e6)  # Large C -> hard margin
svc_2d.fit(X_2d, y_2d)
w_svm = svc_2d.coef_[0]
b_svm = svc_2d.intercept_[0]

margins = functional_margin(X_2d, y_2d, w_svm, b_svm)
support_vectors = X_2d[np.abs(margins - 1.0) < 0.05]  # Points on margin boundaries

print(f"Weight vector ||w||: {np.linalg.norm(w_svm):.4f}")
print(f"Margin width (2/||w||): {margin_width(w_svm):.4f}")
print(f"Number of support vectors: {len(svc_2d.support_vectors_)}")
print(f"Min functional margin (should be ~1): {margins.min():.4f}")
print(f"Max functional margin: {margins.max():.4f}")

# Demonstrate that wider margin = better generalization
for scale in [0.5, 1.0, 2.0]:
    w_test = w_svm * scale
    print(f"Scale={scale:.1f} | margin={margin_width(w_test):.4f} | "
          f"||w||={np.linalg.norm(w_test):.4f}")

## Level 2: SVC Kernel Comparison — Linear/RBF/Poly with OOM Handling

In [ ]:
# Kernel trick: map features to higher dimension implicitly.
# Linear: K(x,z) = x^T z. RBF: K(x,z) = exp(-gamma||x-z||^2). Poly: (x^T z + r)^d.
# Complexity: SVC is O(n^2..n^3) in training — use LinearSVC or SGD for large n.

from sklearn.model_selection import train_test_split

# Use make_moons for non-linear decision boundary
X_moons, y_moons = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(
    X_moons, y_moons, test_size=0.25, random_state=0
)

kernel_configs = [
    ('linear',  {'kernel': 'linear',  'C': 1.0}),
    ('rbf',     {'kernel': 'rbf',     'C': 1.0, 'gamma': 'scale'}),
    ('poly-3',  {'kernel': 'poly',    'C': 1.0, 'degree': 3, 'coef0': 1}),
    ('poly-5',  {'kernel': 'poly',    'C': 1.0, 'degree': 5, 'coef0': 1}),
]

print(f"{'Kernel':>9} | {'Train Acc':>10} | {'Test Acc':>9} | {'n_support_vecs':>14}")
print("-" * 52)
sc = StandardScaler()
X_tr_ms = sc.fit_transform(X_tr_m)
X_te_ms = sc.transform(X_te_m)

for name, kwargs in kernel_configs:
    try:
        clf = SVC(**kwargs)
        clf.fit(X_tr_ms, y_tr_m)
        train_acc = clf.score(X_tr_ms, y_tr_m)
        test_acc = clf.score(X_te_ms, y_te_m)
        n_sv = len(clf.support_vectors_)
        print(f"{name:>9} | {train_acc:10.3f} | {test_acc:9.3f} | {n_sv:14d}")
    except MemoryError:
        print(f"{name:>9} | OOM — kernel matrix too large; use LinearSVC or approximate RBF")

# For large datasets: use LinearSVC or kernel approximation
print("\nScaling tip: LinearSVC is O(n) in time — use for n > 50K samples.")
linear_svc = LinearSVC(C=1.0, max_iter=1000)
linear_svc.fit(X_tr_ms, y_tr_m)
print(f"LinearSVC test acc: {linear_svc.score(X_te_ms, y_te_m):.3f} (fast but linear only)")

## Real-World Example 1: SVM Text Classification with TF-IDF

In [ ]:
# SVM + TF-IDF is a classic, competitive text classification baseline.
# Works well in high-dimensional sparse feature spaces where linear kernel shines.
# In practice: outperforms many neural models on small datasets (<5K samples).

# Synthetic text data: simulate review-like documents
POSITIVE_WORDS = ['great', 'excellent', 'good', 'awesome', 'love', 'perfect', 'best',
                  'wonderful', 'amazing', 'fantastic', 'happy', 'enjoy', 'recommend']
NEGATIVE_WORDS = ['bad', 'terrible', 'awful', 'hate', 'poor', 'worst', 'horrible',
                  'disappoint', 'waste', 'boring', 'mediocre', 'avoid', 'regret']
NEUTRAL_WORDS = ['the', 'a', 'is', 'was', 'this', 'product', 'item', 'service',
                 'experience', 'quality', 'price', 'value', 'delivery', 'bought']


def generate_review(label: int, length: int = 20, seed: int = 0) -> str:
    """Simulate a review with word distribution based on sentiment."""
    rng = np.random.default_rng(seed)
    signal_words = POSITIVE_WORDS if label == 1 else NEGATIVE_WORDS
    n_signal = int(0.3 * length)
    n_neutral = length - n_signal
    words = (
        list(rng.choice(signal_words, n_signal)) +
        list(rng.choice(NEUTRAL_WORDS, n_neutral))
    )
    rng.shuffle(words)
    return ' '.join(words)


N_REVIEWS = 2000
reviews = [generate_review(i % 2, seed=i) for i in range(N_REVIEWS)]
labels = [i % 2 for i in range(N_REVIEWS)]

split = 1600
train_reviews, test_reviews = reviews[:split], reviews[split:]
y_tr_text, y_te_text = labels[:split], labels[split:]

# TF-IDF + SVM pipeline
text_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000, sublinear_tf=True)),
    ('clf', LinearSVC(C=1.0, max_iter=1000)),  # LinearSVC for sparse high-dim features
])
text_pipeline.fit(train_reviews, y_tr_text)
test_acc = text_pipeline.score(test_reviews, y_te_text)

# Feature weights: highest = most discriminative n-grams
vocab = text_pipeline.named_steps['tfidf'].vocabulary_
weights = text_pipeline.named_steps['clf'].coef_[0]
top_positive = sorted(vocab.items(), key=lambda x: weights[x[1]], reverse=True)[:5]
top_negative = sorted(vocab.items(), key=lambda x: weights[x[1]])[:5]

print(f"TF-IDF + LinearSVC test accuracy: {test_acc:.3f}")
print("Top positive features:", [w for w, _ in top_positive])
print("Top negative features:", [w for w, _ in top_negative])

## Real-World Example 2: Kernel Trick — 2D to 3D Feature Visualization

In [ ]:
# The kernel trick implicitly maps data to higher-dimensional space.
# For XOR-like problems: not linearly separable in 2D, but linearly separable in 3D.

# Generate XOR data (classic non-linear problem)
rng = np.random.default_rng(7)
n_each = 100
X_xor = rng.standard_normal((n_each * 4, 2))
y_xor = np.array(
    [1 if x[0]*x[1] > 0 else 0 for x in X_xor]
)

# Try linear SVM — fails on XOR
svc_lin_xor = SVC(kernel='linear', C=1.0)
scores_lin = cross_val_score(svc_lin_xor, X_xor, y_xor, cv=5)

# RBF SVM — succeeds via implicit feature map
svc_rbf_xor = SVC(kernel='rbf', C=10.0, gamma='scale')
scores_rbf = cross_val_score(svc_rbf_xor, X_xor, y_xor, cv=5)

print(f"Linear SVM on XOR: {scores_lin.mean():.3f} +/- {scores_lin.std():.3f}")
print(f"RBF SVM on XOR:    {scores_rbf.mean():.3f} +/- {scores_rbf.std():.3f}")

# Explicit polynomial feature map: phi(x1, x2) = [x1, x2, x1^2, x2^2, sqrt(2)*x1*x2]
# This is what the polynomial kernel computes implicitly.
def poly_feature_map(X: np.ndarray) -> np.ndarray:
    """Explicit degree-2 polynomial feature map for 2D inputs."""
    x1, x2 = X[:, 0], X[:, 1]
    return np.column_stack([x1, x2, x1**2, x2**2, np.sqrt(2)*x1*x2])

X_xor_poly = poly_feature_map(X_xor)
svc_lin_poly = SVC(kernel='linear', C=1.0)
scores_poly = cross_val_score(svc_lin_poly, X_xor_poly, y_xor, cv=5)
print(f"Linear SVM on phi(XOR): {scores_poly.mean():.3f} +/- {scores_poly.std():.3f}")
print(f"\nKey insight: linear in 5D feature space = non-linear in original 2D space.")
print(f"Kernel trick avoids explicit feature map (expensive for high-degree polynomials).")

## Real-World Example 3: SVM vs Logistic Regression in High Dimensions

In [ ]:
# In high-dimensional spaces (text, genomics), linear models dominate.
# SVM and LR both find linear boundaries, but optimize different objectives:
# SVM: hinge loss + max margin. LR: log-loss + regularization.
# SVM is better when classes are close (small margin); LR better for probability estimates.

from sklearn.model_selection import train_test_split
import time

print(f"{'n_features':>12} | {'LR acc':>8} | {'SVM acc':>9} | {'LR time':>9} | {'SVM time':>9}")
print("-" * 58)

for n_features in [50, 200, 1000, 5000]:
    X_hd, y_hd = make_classification(
        n_samples=1000, n_features=n_features,
        n_informative=max(10, n_features // 10),
        n_redundant=0, random_state=42
    )
    X_tr_hd, X_te_hd, y_tr_hd, y_te_hd = train_test_split(
        X_hd, y_hd, test_size=0.25, random_state=0
    )
    sc_hd = StandardScaler()
    X_tr_hd_s = sc_hd.fit_transform(X_tr_hd)
    X_te_hd_s = sc_hd.transform(X_te_hd)

    t0 = time.perf_counter()
    lr_hd = LogisticRegression(max_iter=500, C=1.0, solver='saga').fit(X_tr_hd_s, y_tr_hd)
    lr_time = time.perf_counter() - t0
    lr_acc = lr_hd.score(X_te_hd_s, y_te_hd)

    t0 = time.perf_counter()
    svm_hd = LinearSVC(C=1.0, max_iter=1000).fit(X_tr_hd_s, y_tr_hd)
    svm_time = time.perf_counter() - t0
    svm_acc = svm_hd.score(X_te_hd_s, y_te_hd)

    print(f"{n_features:>12} | {lr_acc:8.3f} | {svm_acc:9.3f} | "
          f"{lr_time:9.3f}s | {svm_time:9.3f}s")

print("\nWhen to use SVM vs LR:")
print("  SVM: high-dimensional, sparse, small datasets. Kernel SVM for non-linear.")
print("  LR: need calibrated probabilities, large n, interpretability required.")